In [68]:
import pandas as pd
import numpy as np
import requests
import io

## Valid vs ignored pollutants

In [69]:
VALID_POLLUTANTS = {"O3", "NO2", "CO", "PM10", "PM2.5", "SO2"}
IGNORED_POLLUTANTS = {"NO", "NOX"}

## Robust reader for NAPS hourly CSVs

In [78]:
BASE_URL = "https://data-donnees.az.ec.gc.ca/download?filename="

def read_naps_csv(filename):
    url = BASE_URL + filename

    df = pd.read_csv(
        url,
        skiprows=7,     # header is on line 8
        sep="\t"        # TAB-separated (as in your screenshot)
    )

    # Clean bilingual column names (keep left side)
    df.columns = [c.split("//")[0].strip() for c in df.columns]

    return df


In [79]:
df = read_naps_csv("SO2_2024.csv")
print(df.columns)
print(df.head())

Index(['<meta name="viewport" content="width=device-width, initial-scale=1.0" />'], dtype='str')
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
0      <link rel="icon" type="image/x-icon" href=...                      
1                                <!-- Open Graph -->                      
2      <meta property="og:type" content="website" />                      
3                                              <meta                      
4                                property="og:title"                      


## Reshape hourly → long format

In [60]:
def load_hourly_naps(df):

    # Normalize pollutant name
    df["Pollutant"] = (
        df["Pollutant"]
        .str.replace("PM25", "PM2.5")
        .str.strip()
    )

    # Ignore NO and NOX
    if df["Pollutant"].iloc[0] in {"NO", "NOX"}:
        return None

    hour_cols = [c for c in df.columns if c.startswith("H")]

    df_long = df.melt(
        id_vars=[
            "Pollutant",
            "NAPS ID",
            "City",
            "Province",
            "Latitude",
            "Longitude",
            "Date"
        ],
        value_vars=hour_cols,
        var_name="Hour",
        value_name="Concentration"
    )

    df_long["Hour"] = df_long["Hour"].str.replace("H", "").astype(int) - 1
    df_long["Datetime"] = (
        pd.to_datetime(df_long["Date"]) +
        pd.to_timedelta(df_long["Hour"], unit="h")
    )

    df_long.loc[df_long["Concentration"] == -999, "Concentration"] = np.nan

    return df_long


## Unit conversion safeguard (O₃ ppb → ppm)

In [61]:
def apply_unit_conversion(df):
    if df["Pollutant"].iloc[0] == "O3":
        df["Concentration"] = df["Concentration"] / 1000.0  # ppb → ppm
    return df

## Daily AQI concentration metric

In [62]:
def daily_metric(df):
    pol = df["Pollutant"].iloc[0]
    g = df.groupby(["NAPS ID", df["Datetime"].dt.date])

    if pol in {"O3", "CO"}:
        return (
            g.apply(lambda x:
                x.set_index("Datetime")["Concentration"]
                 .rolling(8, min_periods=6).mean().max()
            )
            .rename("C")
            .reset_index()
        )

    if pol in {"NO2", "SO2"}:
        return g["Concentration"].max().reset_index(name="C")

    if pol in {"PM10", "PM2.5"}:
        return g["Concentration"].mean().reset_index(name="C")

## EPA AQI breakpoints (Table 6, 2024)

In [54]:
AQI_BREAKPOINTS = {
    "CO": [(0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
           (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500)],
    "NO2": [(0,53,0,50),(54,100,51,100),(101,360,101,150),
            (361,649,151,200),(650,1249,201,300),(1250,2049,301,500)],
    "SO2": [(0,35,0,50),(36,75,51,100),(76,185,101,150),
            (186,304,151,200),(305,604,201,300),(605,1004,301,500)],
    "PM2.5": [(0,9.0,0,50),(9.1,35.4,51,100),(35.5,55.4,101,150),
              (55.5,125.4,151,200),(125.5,225.4,201,300),(225.5,325.4,301,500)],
    "PM10": [(0,54,0,50),(55,154,51,100),(155,254,101,150),
             (255,354,151,200),(355,424,201,300),(425,604,301,500)],
    "O3": [(0.000,0.054,0,50),(0.055,0.070,51,100),
           (0.071,0.085,101,150),(0.086,0.105,151,200),
           (0.106,0.200,201,300)]
}

##  AQI calculation function

In [64]:
def calc_aqi(C, pollutant):
    if pd.isna(C):
        return np.nan

    for bp_lo, bp_hi, I_lo, I_hi in AQI_BREAKPOINTS[pollutant]:
        if bp_lo <= C <= bp_hi:
            return round(
                (I_hi - I_lo) / (bp_hi - bp_lo) * (C - bp_lo) + I_lo
            )

    return np.nan

## Run AQI calculation for all 2024 NAPS pollutants

In [65]:
FILES = {
    "O3":  "O3_2024.csv",
    "NO2": "NO2_2024.csv",
    "CO":  "CO_2024.csv",
    "PM2.5": "PM25_2024.csv",
    "PM10": "PM10_2024.csv",
    "SO2": "SO2_2024.csv",
    "NO":  "NO_2024.csv",
    "NOX": "NOX_2024.csv"
}

BASE_URL = (
    "https://data-donnees.az.ec.gc.ca/data/air/monitor/"
    "national-air-pollution-surveillance-naps-program/"
    "Data-Donnees/2024/ContinuousData-DonneesContinu/"
    "HourlyData-DonneesHoraires/"
)

daily_results = []

for pol, fname in FILES.items():
    print(f"\nProcessing {pol}")

    df_raw = read_naps_csv(BASE_URL + fname)
    df_long = load_hourly_naps(df_raw)

    if df_long is None:
        print("  → skipped (NO / NOX)")
        continue

    df_long = apply_unit_conversion(df_long)
    daily = daily_metric(df_long)

    daily["Pollutant"] = pol
    daily["AQI"] = daily["C"].apply(lambda c: calc_aqi(c, pol))

    daily_results.append(daily)
    print("  → added", len(daily), "daily values")

aqi_daily = pd.concat(daily_results, ignore_index=True)



Processing O3


KeyError: 'Pollutant'